# Stateful Agent — Week 4
Plan → Human Approval (HITL) → Execute → Stream → Persist (SQLite) → LangSmith Trace

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path="/Users/revathy_santhosh/stateful_agent/.env")

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

print("LLM ready.")

LLM ready.


## 2. Define State and Tool

In [2]:
import ast
import operator
from typing import TypedDict, Annotated
from langchain_core.tools import tool
from langgraph.graph.message import add_messages

# Agent's "memory" shape as it moves through the graph
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    plan: str
    approved: bool

# Same safe calculator tool from Week 3
_ops = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ops[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ops[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '23 * 47 + 19'."""
    tree = ast.parse(expression, mode="eval")
    return str(_safe_eval(tree.body))

print("State and tool defined.")

State and tool defined.


## 3. Planner Node

In [3]:
def planner_node(state: AgentState):
    user_msg = state["messages"][-1].content
    prompt = (
        f"A user asked: '{user_msg}'. "
        "Reply with ONLY a one-sentence plan describing the exact calculation "
        "you will run using the calculator tool. Do not solve it yourself."
    )
    response = llm.invoke(prompt)
    return {"plan": response.content, "messages": [response]}

print("Planner node defined.")

Planner node defined.


## 4. Human Approval Node (HITL)

In [4]:
from langgraph.types import interrupt

def human_approval_node(state: AgentState):
    decision = interrupt(
        {
            "question": "Approve this plan? (yes/no)",
            "plan": state["plan"],
        }
    )
    return {"approved": decision}

print("Human approval node defined.")

Human approval node defined.


## 5. Executor Node

In [5]:
from langchain_core.messages import AIMessage, HumanMessage

llm_with_tools = llm.bind_tools([calculator])

def executor_node(state: AgentState):
    if state.get("approved") != "yes":
        return {"messages": [AIMessage(content="Plan was not approved. Stopping here.")]}

    ai_msg = llm_with_tools.invoke([
        HumanMessage(content=(
            f"Execute this plan using a SINGLE calculator tool call with the full, "
            f"complete expression (not just part of it): {state['plan']}"
        ))
    ])
    results = []
    for call in ai_msg.tool_calls:
        if call["name"] == "calculator":
            result = calculator.invoke(call["args"])
            results.append(f"{call['args']['expression']} = {result}")

    summary = "\n".join(results) if results else "No tool calls were made."
    return {"messages": [AIMessage(content=summary)]}

print("Executor node updated.")

Executor node updated.


## 6. Build the Graph

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

graph = StateGraph(AgentState)
graph.add_node("planner", planner_node)
graph.add_node("human_approval", human_approval_node)
graph.add_node("executor", executor_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "human_approval")
graph.add_edge("human_approval", "executor")
graph.add_edge("executor", END)

app = graph.compile(checkpointer=checkpointer)

print("Graph compiled with SQLite persistence.")

Graph compiled with SQLite persistence.


## 7. Run the Agent — First Pause

In [7]:
config = {"configurable": {"thread_id": "demo-run-1"}}

result = app.invoke(
    {"messages": [("user", "What is 23 times 47, plus 19?")], "plan": "", "approved": False},
    config=config,
)

print(result)

{'messages': [HumanMessage(content='What is 23 times 47, plus 19?', additional_kwargs={}, response_metadata={}, id='f8b96aae-bcd8-427b-8398-8a7307bf798b'), AIMessage(content='I will calculate 23 times 47 plus 19 by first multiplying 23 by 47, then adding 19 to the product.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 174, 'prompt_tokens': 115, 'total_tokens': 289, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 137, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.083830316, 'prompt_time': 0.006501681, 'completion_time': 0.182167118, 'total_time': 0.188668799}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_aef561cf8e', 'id': 'chatcmpl-6fdc99ff-078c-45df-a7f7-62cf920fde15', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1fe42-95c5-7ae2-ad9f

In [8]:
print("PLAN:", result.get("plan"))
print("INTERRUPT INFO:", result.get("__interrupt__"))

PLAN: Multiply 23 by 47, then add 19 to that product.
INTERRUPT INFO: [Interrupt(value={'question': 'Approve this plan? (yes/no)', 'plan': 'Multiply 23 by 47, then add 19 to that product.'}, id='d3713901d6d015bce17bdb48e540bbd9', response_schema=None)]


## 8. Approve and Resume

In [9]:
from langgraph.types import Command

result2 = app.invoke(Command(resume="yes"), config=config)
print(result2["messages"][-1].content)

23 * 47 + 19 = 1100


In [10]:
config3 = {"configurable": {"thread_id": "demo-run-3"}}

result4 = app.invoke(
    {"messages": [("user", "What is 100 divided by 4?")], "plan": "", "approved": False},
    config=config3,
)
print(result4.get("plan"))

# Now reject it
result5 = app.invoke(Command(resume="no"), config=config3)
print(result5["messages"][-1].content)

I will enter “100 ÷ 4” into the calculator and retrieve the result.
Plan was not approved. Stopping here.


## 9. Streaming

In [11]:
config4 = {"configurable": {"thread_id": "demo-run-4"}}

print("--- First run (will pause for approval) ---")
for event in app.stream(
    {"messages": [("user", "What is 15 plus 85?")], "plan": "", "approved": False},
    config=config4,
    stream_mode="updates",
):
    print(event)

print("\n--- Resuming after approval ---")
for event in app.stream(Command(resume="yes"), config=config4, stream_mode="updates"):
    print(event)

--- First run (will pause for approval) ---
{'planner': {'plan': 'I will use the calculator tool to add 15 and 85.', 'messages': [AIMessage(content='I will use the calculator tool to add 15 and 85.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 111, 'total_tokens': 226, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 92, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.145032134, 'prompt_time': 0.006139312, 'completion_time': 0.12123818, 'total_time': 0.127377492}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'id': 'chatcmpl-a2916d68-d089-487a-8776-d5ee66954587', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20d85-d32b-71b1-a344-a7e2e63ad0f7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input